<div>
<center><img src="assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Quantum Devices as Flux Resources</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">QFw under Flux, on your laptop, with simulators</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">October 2026</p>
</div>

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Open a terminal.</span> This notebook is for reading. You run every command in a terminal. In JupyterLab choose <b>File, New, Terminal</b>, then drag the terminal tab to the right so it sits next to this notebook. Copy each command from here and paste it there.
</div>

A quantum computer is a resource like any other. That is the whole idea of this tutorial. We put each quantum device into the Fluxion resource graph next to the compute nodes. Then one Flux job asks for nodes and a device together. Fluxion hands out both at once, or makes the job wait, the same way it would for a busy core.

The devices here are simulators and a fake vendor queue, so nothing leaves your laptop. The software that fronts a device is called a QPM. You will meet it in a minute.

## Before you start

You are inside the tutorial container, and a Flux instance with nine brokers is running around you. Rank 0 is `flux-lead` and the other eight are `c1` through `c8`. They behave like nine separate nodes, all inside this one container. Your terminal is already connected to that instance.

If you would rather watch it all run, `./demo/run-all.sh` plays every part of this notebook in the terminal, with the same commentary, pausing before each command.

Check that it can see Flux.

```bash
flux getattr size
flux resource list -s all -no '{nodelist}'
```

You should see `9`, then `c[1-8]`.

We will list jobs a lot, so the container has a short command for it called `qjobs`. It shows only the jobs that are pending or running, with their names and nodes. It is the same as this.

```bash
flux jobs -o '{id.f58:>12} {name:<14} {state:<6} {nnodes:>2} {nodelist}'
```

## Start fresh

If you ran this before, there may be jobs still sleeping in the queue. This cancels them, waits until nothing is active, and clears finished jobs out of the history. You never need to restart the container.

```bash
flux cancel --all --user=all
flux queue idle --timeout=60
flux job purge --force --num-limit=0
qjobs
```

<br>

## Part 1. The resource graph

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600">Look at the nodes first.</span> This is a plain Fluxion view, and there is nothing quantum about it yet.
</div>

```bash
flux resource list
```

Eight nodes with four cores each, all in the `normal` queue. The `flux-lead` node runs the broker and is kept out of scheduling.

### The quantum devices

Now the part that is new. Each QPM shows up as a vertex in the same graph.

```bash
flux-qfw devices
```

`qpus` is how many jobs may hold the device at the same time, and `qubits` is how wide the device is. `fake-iqm` allows two, and the rest allow one.

Here is the same thing straight from the graph.

```bash
flux kvs get resource.R | jq -c '[.scheduling.graph.nodes[].metadata | select(.type=="qpu") | .paths.containment]'
```

Each `qpu` has a path like `/cluster0/qdevice_nwqsim0/qpu0`. The devices sit under the cluster root, beside the nodes and not under any node, because any node can reach a device over the network.

### What a hybrid request looks like

The `flux-qfw` plugin adds a flag called `--qfw-qpu`. It adds a second entry to the resources the job asks for, and Fluxion matches the nodes and the device in a single pass. A dry run shows the request without submitting anything.

```bash
flux run --dry-run -N2 -n8 --qfw-qpu=nwqsim --qfw-max-qubits=5 hostname | jq -c '.resources[1], (.attributes.system.qfw | {qpu, scope, queue, max_qubits})'
```

The first line is the device entry. Its `qpu` asks for five qubits, so Fluxion only matches a device at least that wide, and a job that asks for more qubits than the device has is refused at submit. The second line is the envelope, the size of the work, which also goes to the QPM.

<br>

## Part 2. A device the site owns

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600">The point of this part.</span> When the site controls a device, a grant from Fluxion means the device is ready. Nodes and device are handed out together.
</div>

`nwqsim` is a simulator that runs on the job's own nodes. Nobody else can be using it, so if Fluxion gives it to you, it is yours right now.

### One hybrid job runs

The nodes and the `nwqsim` qpu are free, so this starts right away. We keep the job id in a shell variable for later.

```bash
A=$(flux submit --job-name=A-nwqsim -N2 -n8 --qfw-qpu=nwqsim sleep 300)
qjobs
```

### A second job on the same device waits

```bash
B=$(flux submit --job-name=B-nwqsim -N1 -n4 --qfw-qpu=nwqsim sleep 300)
qjobs
```

`nwqsim` has one qpu and job A holds it, so job B sits in `SCHED`. There are plenty of free cores. B is not waiting on cores. It is waiting on the device, exactly the way it would wait on a busy core.

### Classical work is not blocked

A job that does not ask for a device packs in around A and B, like on any Flux cluster.

```bash
F=$(flux submit --job-name=F-classical -N1 -n1 sleep 300)
qjobs
```

### The running job's R carries the qpu

Every Flux job has its own R, which is the set of resources it was granted.

```bash
flux job info $A R | jq -c '{nodes: .execution.nodelist, qpu: [.scheduling.graph.nodes[].metadata | select(.type=="qpu") | .paths.containment]}'
```

Job A's R names its two nodes and the qpu. The device is part of the grant, not a side channel.

### Freeing the device starts the waiting job

```bash
flux cancel $A
qjobs
```

Fluxion gives the qpu to B at once, and B goes from `SCHED` to `RUN`. The device that looked stuck was only busy.

<br>

## Part 3. A device behind a vendor queue

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600">The point of this part.</span> When somebody else controls the device, a grant from Fluxion does not mean the device is ready. So the submit becomes a pair. A scout job holds the device in the graph and talks to the vendor, and your job is held until the scout says go.
</div>

`fake-iqm` stands in for a real machine at a vendor. The site does not control it. If Fluxion just handed out the nodes, your job would start, sit there, and wait on the vendor while the nodes did nothing.

So this device is marked `queue = "external"` in `conf/qfw.toml`. You type the same command, but it submits two jobs, in order. First the plugin checks there is room for your job plus one core for a scout. **Your job** goes in first, **held**, and asks for no device at all. Fluxion keeps its nodes reserved, but nothing starts on them. Then the **scout** goes in, naming your job. It asks for one core and the qpu, so it waits on the device like any other job and only runs while it holds a lease.

### Submit the pair

Your job prints its reservation and then sleeps for five minutes, so you have time to look around. The envelope flags matter here, because the scout passes them to the vendor.

```bash
C=$(flux submit --job-name=C-fake-iqm -N1 -n4 --qfw-qpu=fake-iqm --qfw-max-qubits=5 --qfw-max-shots=1024 \
    sh -c 'echo QFW_RESERVATIONS=$QFW_RESERVATIONS; sleep 300')
qjobs
```

The plugin printed `scout job ... submitted for fake-iqm`, and two jobs appeared. `scout:fake-iqm` is running because a lease was free. `C-fake-iqm` is in `SCHED` with free nodes. That is the hold, not a busy device. Look at how it was submitted.

```bash
flux job info $C jobspec | jq -c '.attributes.system | {hold, pair: .qfw.pair}'
flux job id --to=f58 $(flux jobs -no '{user.qfw_scout}' $C)
```

### The scout gets a session and releases your job

The scout runs as you. It opens a session on the vendor's queue, waits for it, writes the reservation onto your job as a memo, and then releases the hold. The fake vendor takes about twenty seconds. Run this right away, and again a few times, and watch it change from queued to running.

```bash
flux-qfw sessions
```

Now wait for your job to start, and look at the memos on it.

```bash
flux job wait-event -t 60 $C start
flux job eventlog $C | grep memo
qjobs
```

There are three memos. The first names the scout, the second says a session was opened, and the third carries the reservation and a `released` key, which is the moment the hold was cleared. Your job started right after that. The scout keeps running beside it, holding the device and the session for as long as your job runs.

### Your job got the scout's reservation

Your job is still running. Its output so far is kept in an eventlog, which you can read without waiting for the job to end.

```bash
flux job eventlog -p guest.output $C | grep -E 'QFW_RESERVATIONS|flux-qfw'
```

The first line is the job shell picking up the reservation the scout memoed. Then each of the four tasks printed the same `QFW_RESERVATIONS`. That is the session the scout got before your job ran. The shell reserved nothing and will release nothing.

Your job's R has no qpu in it. The scout's R does, which is where it belongs.

```bash
flux job info $C R | jq '[.scheduling.graph.nodes[].metadata.type | select(. == "qpu")] | length'
S=$(flux jobs -no '{user.qfw_scout}' $C)
flux job info $S R | jq '[.scheduling.graph.nodes[].metadata.type | select(. == "qpu")] | length'
```

You should see `0` for your job and `1` for the scout.

### When your job ends, the scout lets go

The scout holds the device and the session for exactly as long as your job runs. End your job and watch the scout follow it.

```bash
flux cancel $C
sleep 5
qjobs
flux-qfw sessions
flux job attach $S
```

Both jobs are gone and no pairs are left. The scout's own log tells the story. It found your job, opened a session, released your job when the session was ready, and closed the session when your job ended.

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Why this matters.</span> On a real vendor machine the wait can be long. Holding your job means the classical nodes are not started early and left idle. The scout holding the device means only as many vendor sessions are open as the device has leases. When the vendor is ready, the nodes and the device begin together.
</div>

<br>

## Part 4. What the job sees

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600">Two scopes.</span> A job scope device is run by the job itself. A site scope device is a service the site runs, and the job registers with it.
</div>

### Job scope

This is the default for the simulators here. There is no reservation at all.

```bash
flux run -N2 -n2 --qfw-qpu=shim-ornl-20q sh -c 'echo rank $FLUX_TASK_RANK: scope=$QFW_QPU_SCOPE nodes=$QFW_SIMULATOR_NODES agent=$(basename $PRTE_MCA_plm_ssh_agent)'
```

Every rank learns the job's own hosts, which is where the simulator will run. It also gets a launch agent, so the MPI runtime starts its daemons with `flux exec` instead of ssh. QFw then starts what it needs on those hosts, as you, and tears it down when the job ends.

### Site scope

Here we force site scope on the same device.

```bash
flux run -N1 -n1 --qfw-qpu=shim-ornl-20q --qfw-scope=site --qfw-max-qubits=3 sh -c 'echo QFW_RESERVATIONS=$QFW_RESERVATIONS'
```

The job already holds the device, so the shell does not acquire anything. It tells the QPM that the job has its device when the job starts, and tells it again when the job exits. The QPM hands back the reservation id QFw needs. You can see both steps in the two `flux-shell` lines around the output. This goes through the mock backend, so nothing real is contacted.

### An allocation registers once

If you have used `salloc --qpu` on Slurm, this is the same idea.

```bash
flux alloc -N1 -n2 --qfw-qpu=iqm-ornl-20q --qfw-scope=site --qfw-max-qubits=5 --qfw-max-shots=16
```

That gives you a shell inside the allocation. Run two steps there. Neither asks for the device.

```bash
flux run -n2 sh -c 'echo step 1: $QFW_RESERVATIONS'
flux run -n1 sh -c 'echo step 2: $QFW_RESERVATIONS'
exit
```

Both steps print the same value. They inherit it from the allocation's shell, which registered once when it started.

## Clean up

Cancel the sleeps so the cluster is quiet again.

```bash
flux cancel --all --user=all
flux queue idle --timeout=60
qjobs
```

<br>

## 🧭 Side quests

<div class="alert alert-block" style="background-color:#D9A441;color:#06293D">
<span style="font-weight:600">Finished early?</span> Pick one. Each takes about ten minutes, and an answer is folded up underneath it. Try before you unfold.
</div>

### 1. Three pairs, two leases

`fake-iqm` allows two jobs at a time. Submit three jobs on it and watch `qjobs` and `flux-qfw sessions`. How many scouts run, and how many sessions get opened?

<details><summary>One answer</summary>

```bash
for i in 1 2 3; do flux submit --job-name=q$i -N1 -n2 --qfw-qpu=fake-iqm sleep 60; done
sleep 8
qjobs
flux-qfw sessions
```

Two scouts run and the third waits in `SCHED` on the device, the same way job B waited on `nwqsim`. Only two sessions are ever open, because a scout can only talk to the vendor while it holds a lease. The third user job stays held until its scout gets a turn.
</details>

### 2. Cancel while held

Submit a job on `fake-iqm` and cancel it before the vendor is ready. What happens to the scout and the session?

<details><summary>One answer</summary>

```bash
X=$(flux submit --job-name=cancelme -N1 -n1 --qfw-qpu=fake-iqm sleep 60)
S=$(flux jobs -no '{user.qfw_scout}' $X)
sleep 1; flux cancel $X
flux job wait-event -t 30 $S clean
flux job attach $S
```

The scout notices its partner is gone, closes the session, and exits on its own. Nothing is left open at the vendor and nothing is left in the queue.
</details>

### 3. Count the graph

Pull the whole scheduling graph out of the KVS and count the vertices by type. How many cores, qpus and qubit pools are there, and does the count match `flux resource list` and `flux-qfw devices`?

<details><summary>One answer</summary>

```bash
flux kvs get resource.R | jq -r '.scheduling.graph.nodes[].metadata.type' | sort | uniq -c
```

Thirty six cores across nine nodes. Four of those cores are on `flux-lead`, which is kept out of scheduling, so `flux resource list` shows thirty two. There are five qpus across four devices, and each qpu has one qubit pool as wide as its device. The extra qpu is the second lease on `fake-iqm`.
</details>

<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Tutorial complete</strong></div>

The pieces you used are the `hold` branch of [flux-sched](https://github.com/vsoch/flux-sched), which adds the hold and the `coschedule` policy that keeps a held job's nodes reserved, the [flux-qfw](https://github.com/converged-computing/flux-qfw) plugins that add `--qfw-qpu`, and [QFw](https://github.com/openQSE/QFw) itself. The `flux/` directory in this repository wires them together.